## Tratamiento de ofertas laborales


In [ ]:
# !pip install pandas nltk spacy
# !python -m spacy download es_core_news_sm

## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter
from pathlib import Path

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

## 2 · Carga de datos

In [ ]:
df_bumeran      = pd.read_csv("empleos_bumeran.csv", encoding="utf-8")
df_computrabajo = pd.read_csv( "trabajos_computrabajo.csv", encoding="utf-8")

# Unificar columnas
df_bumeran = df_bumeran[["titulo", "detalle", "localizacion", "portal"]]
df_bumeran = df_bumeran.rename(columns={"localizacion": "ubicacion"})

def limpiar_titulo_computrabajo(texto):
    """Elimina prefijos 'Empleo de' / 'Trabajos de' de Computrabajo."""
    if isinstance(texto, str):
        for prefijo in ["Empleo de ", "Trabajos de "]:
            if texto.startswith(prefijo):
                return texto[len(prefijo):].strip()
    return texto

df_computrabajo = df_computrabajo[["titulo", "detalle", "portal", "ubicacion"]].copy()
df_computrabajo["titulo"] = df_computrabajo["titulo"].apply(limpiar_titulo_computrabajo)

df_total = pd.concat([df_bumeran, df_computrabajo], ignore_index=True)
df_total.head(2)

,titulo,detalle,ubicacion,portal
0,Analista de Datos,¡Sé parte de Stefanini!​En Stefanini somos más...,"San Borja, Lima",bumeran
1,Analista de Datos (Educación),"Somos VISIVA (Certus,UCAL y Toulouse Lautrec) ...","San Borja, Lima",bumeran


## 3 · Nivel del puesto

In [ ]:
def clasificar_nivel(texto):
    """Detecta el nivel del puesto en el texto del aviso."""
    t = texto.lower()
    if "practicante" in t:
        return "Practicante"
    elif "senior" in t or " sr " in t or "sr." in t:
        return "Senior"
    elif "junior" in t or " jr " in t or "jr." in t:
        return "Junior"
    return "No especifica"

df_total["Nivel"] = df_total["detalle"].apply(clasificar_nivel)
print(df_total["Nivel"].value_counts())

Nivel
No especifica    1581
Senior            251
Practicante       164
Junior             92
Name: count, dtype: int64


## 4 · Bigrams técnicos

In [ ]:

BIGRAMS_TECNICOS = [
    # BI / Data
    (r'power\s+bi',             'power_bi'),
    (r'business\s+intelligence','business_intelligence'),
    (r'machine\s+learning',     'machine_learning'),
    (r'deep\s+learning',        'deep_learning'),
    (r'data\s+science',         'data_science'),
    (r'big\s+data',             'big_data'),
    (r'inteligencia\s+artificial', 'inteligencia_artificial'),
    # Bases de datos
    (r'bases?\s+de\s+datos',   'bases_datos'),
    (r'sql\s+server',           'sql_server'),
    (r'ms\s+sql',               'ms_sql'),
    (r'no\s*sql',               'nosql'),
    # Cloud / DevOps
    (r'ci\s*/\s*cd',           'ci_cd'),
    (r'google\s+cloud',         'google_cloud'),
    (r'microsoft\s+azure',      'azure'),
    (r'amazon\s+web\s+services','aws'),
    (r'control\s+de\s+versiones','control_versiones'),
    # Office / Excel
    (r'microsoft\s+office',     'ms_office'),
    (r'ms\s+office',            'ms_office'),
    (r'power\s+query',          'power_query'),
    # Otros
    (r'aprendizaje\s+autom[aá]tico', 'machine_learning'),
    (r'procesamiento\s+de\s+lenguaje\s+natural', 'nlp')
]

def proteger_bigrams(texto):
    """Reemplaza bigrams técnicos por versión con _ antes de tokenizar."""
    for patron, reemplazo in BIGRAMS_TECNICOS:
        texto = re.sub(patron, reemplazo, texto, flags=re.IGNORECASE)
    return texto

## 5 · Extracción de la sección de requisitos

In [ ]:
def extraer_requisitos(texto):

    patron_inicio = r"(analista\s+de|requisitos|perfil requerido|lo que buscamos|conocimientos|habilidades)"
    patron_fin    = r"(beneficios|ofrecemos|que brindamos|funciones|responsabilidades)"

    inicio_match = re.search(patron_inicio, texto, re.IGNORECASE)

    if inicio_match:
        texto_recortado = texto[inicio_match.start():]
        fin_match = re.search(patron_fin, texto_recortado, re.IGNORECASE)
        if fin_match and fin_match.start() > 50:
            return texto_recortado[:fin_match.start()]
        return texto_recortado[:800]
    return texto

## 6 · Stopwords ampliadas

In [ ]:
# ── Base NLTK en español ─────────────────────────────────────────────
stop_base = set(nltk_stopwords.words('spanish'))

def quitar_acentos(texto):
    nfkd = unicodedata.normalize('NFKD', str(texto))
    palabra_limpia = ""
    for letra in nfkd:
      if unicodedata.combining(letra) == 0:
          palabra_limpia += letra

    return palabra_limpia

stop_base = {quitar_acentos(w) for w in stop_base}

STOP_VERBOS = {
    'elaborar','garantizar','colaborar','brindar','apoyar',
    'generar','coordinar','gestionar','ejecutar','desarrollar','implementar',
    'realizar','mantener','disenar','participar','construir','monitorear',
    'documentar','proponer','identificar','asegurar','integrar',
    'supervisar','crear','detectar','proveer','soportar',
    'conectar','priorizar','impulsar','facilitar','contribuir',
    'administrar','establecer','definir','evaluar','validar','reportar',
    'comunicar','planificar','liderar','controlar','operar','configurar',

    'realizando','buscando','contribuyendo','aportando','identificando',
    'proponiendo','garantizando','construyendo','orientado','encontramos',

    'busqueda','incorporacion'
}

STOP_BENEFICIOS = {
    # modalidad / horario
    'modalidad','hibrida','presencial','remoto','horario','lunes','viernes',
    'horas','am','pm','turno',
    # compensación
    'cts','gratificaciones','vacaciones','dias','libres','ingreso','sueldo',
    'salario','remuneracion','planilla','ley','beneficios',
    # políticas de empresa
    'discriminacion','inclusion','discapacidad','diversidad','genero',
    'reclutamiento','seleccion','postulantes','retribucion','favores',
    'quot','solicitud','convocatoria','alineada','politicas','promovemos',
    'respetamos','solicitamos','ningun',
    # universidades / lugares (de sección de beneficios)
    'toulouse','lautrec','certus','ucal','becas','corporativas',
    'chacarilla','surco','miraflores','san','isidro',
    # conectores de avisos
    'adicionalmente','asimismo','segun','acuerdo','manera','mediante',
    'parte','areas','grupo','tipo','cuales','oportunidad',
}

STOP_PERFIL = {
    # nivel/experiencia
    'nivel','minimo','deseable',
    'indispensable','experiencia','anos','conocimiento','conocimientos',
    'manejo','dominio','uso',
    # cargo / perfil
    'titulado','egresado','bachiller','profesional','tecnico','analista',
    'ingeniero','licenciado','posiciones','roles','perfil','puesto',
    'especialista','consultor','asistente','jefe','coordinador', "ing",
    # genéricas
    'empresa','oferta','requisitos','equipo','soluciones','importante',
    'encuentra','herramienta','herramientas','capacidad','habilidad',
    'habilidades','competencia','competencias','area','similares',
    'afines','carreras','industriales','disponibilidad',
    # conectores frecuentes en avisos
    'generacion','idealmente','evaluara','proceso','cartera','bancaria',
    'banca','seguros','people','encontramos','orientado','estrategica',
    'contribuyendo','toma','ingresados','pendientes','aicc','debo',
}

STOPWORDS_TOTAL = stop_base | STOP_VERBOS | STOP_BENEFICIOS | STOP_PERFIL

print(f"Stopwords totales: {len(STOPWORDS_TOTAL)}")
print(f"  Base NLTK:    {len(stop_base)}")
print(f"  Verbos:       {len(STOP_VERBOS)}")
print(f"  Beneficios:   {len(STOP_BENEFICIOS)}")
print(f"  Perfil genérico: {len(STOP_PERFIL)}")

Stopwords totales: 491
  Base NLTK:    306
  Verbos:       56
  Beneficios:   65
  Perfil genérico: 67


## 7 · Pipeline de limpieza completo

In [ ]:
def limpiar_texto(texto):

    texto = texto.lower()
    texto = quitar_acentos(texto)
    texto = re.sub(r'[^a-z_\s]', ' ', texto) 
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokenizar(texto, stop):
    tokens = texto.split()
    tokens = [tok for tok in tokens if len(tok) >= 2 and tok not in stop]

    return tokens

def procesar_oferta(texto):
    """Pipeline completo para una oferta:
    1. Extraer sección de requisitos
    2. Proteger bigrams técnicos
    3. Limpiar texto
    4. Tokenizar con stopwords
    """
    texto = extraer_requisitos(texto)
    texto = proteger_bigrams(texto)
    texto = limpiar_texto(texto)
    return tokenizar(texto, STOPWORDS_TOTAL)

# Verificar
ejemplo = df_total['detalle'].iloc[1]
tokens = procesar_oferta(ejemplo)
print(f"Tokens extraídos ({len(tokens)}):")
print(tokens)

Tokens extraídos (63):
['datos', 'tls', 'funciones', 'arquitectura', 'datos', 'soporte', 'operacion', 'educacion', 'virtual', 'integrando', 'fuentes', 'lms', 'crm', 'sistemas', 'academicos', 'ponderar', 'variables', 'criticas', 'ej', 'interaccion', 'calidad', 'feedback', 'tiempos', 'respuesta', 'engagement', 'patrones', 'correlacionen', 'desempeno', 'docente', 'exito', 'alumno', 'explorar', 'procesar', 'informacion', 'cualitativa', 'foros', 'encuestas', 'abiertas', 'feedbacks', 'escritos', 'extraer', 'insights', 'tecnicas', 'analisis', 'texto', 'transformando', 'lenguaje', 'datos', 'accionables', 'tableros', 'dinamicos', 'permitan', 'tiempo', 'real', 'salud', 'aulas', 'virtuales', 'visualizando', 'progreso', 'estudiante', 'cumplimiento', 'docente', 'tendenci']


## 8 · Aplicar a todo el dataset

In [ ]:
df_total['habilidades'] = df_total['detalle'].apply(procesar_oferta)

lens = df_total['habilidades'].map(len)
print(f"Tokens por oferta — promedio: {lens.mean():.1f} | mediana: {lens.median():.0f} | max: {lens.max()}")

Tokens por oferta — promedio: 46.9 | mediana: 38 | max: 598


In [ ]:
filas_vacias = df_total[df_total['habilidades'].str.len() == 0]
for i in filas_vacias["detalle"]:
  print(i)
len(filas_vacias)

Facilitamos un uso inteligente de la tecnología, potenciando a las personas y generando resultados de valor.En Tsoft brindamos asistencia a los máximos responsables de las áreas de IT de nuestros clientes en el cambio organizacional, para lograr mejoras en el gobierno y gestión de sus tecnologías de la información y comunicaciones.Somos más de 1000 personas y conformamos un equipo de profesionales certificados en múltiples tecnologías y con experiencia en diversas metodologías de valor para las áreas clave de IT. Día a día trabajamos estrechamente con nuestros clientes, procurando conocer su negocio y estando atentos a sus necesidades e inquietudes.Fomentamos una actitud de servicio centrada en la empatía, promoviendo el contacto directo y manteniendo una comunicación fluida con ellos.Para mayor información escríbenos a: contacte@tsoftglobal.com Si quieres sumarte a nuestro equipo, envíanos tu CV u hoja de vida a: talento@tsoftlatam.com UI/UX – Frictionless Payments  Build experiences 

1

In [ ]:
df_total = df_total[df_total['habilidades'].map(len) > 0]

In [ ]:

PATRON_PEGADO = re.compile(
    r'(datos|requisitos|formacion|conocimientos|experiencia|modelamiento)'
    r'(datos|requisitos|formacion|conocimientos|experiencia|modelamiento)',
    re.IGNORECASE
)

def limpiar_pegados(tokens):
    limpios = []
    for t in tokens:
        # Si el token es muy largo (>20 chars) probablemente está pegado
        if len(t) > 20:
            continue
        # Si matchea el patrón de dos palabras de relleno pegadas
        if PATRON_PEGADO.search(t):
            continue
        limpios.append(t)
    return limpios

df_total['habilidades'] = df_total['habilidades'].apply(limpiar_pegados)

## 9 · Filtro de frecuencia mínima

In [ ]:
# Contar frecuencia de cada token en todo el corpus
freq_global = Counter()
for toks in df_total['habilidades']:
    freq_global.update(set(toks))  

MIN_FREQ = 2
vocab_valido = {t for t, n in freq_global.items() if n >= MIN_FREQ}

print(f"Vocabulario antes del filtro: {len(freq_global)}")
print(f"Vocabulario después : {len(vocab_valido)}")

eliminados = [t for t in freq_global if t not in vocab_valido]

Vocabulario antes del filtro: 7064
Vocabulario después : 4486


In [ ]:
eliminados[:20]

['feedbacks',
 'aulas',
 'correlacionen',
 'texto',
 'cualitativa',
 'alumno',
 'foros',
 'visualizando',
 'ponderar',
 'escritos',
 'tendenci',
 'trabaja',
 'fc',
 'daras',
 'obtenida',
 'estableceras',
 'contactaras',
 'corto',
 'analizaras',
 'garanticen']

In [ ]:
df_total['habilidades'] = df_total['habilidades'].apply(
    lambda toks: [t for t in toks if t in vocab_valido]
)

lens = df_total['habilidades'].map(len)
print(f"Después del filtro — promedio: {lens.mean():.1f} | mediana: {lens.median():.0f}")
print(f"Ofertas con 0 tokens: {(lens == 0).sum()}")

Después del filtro — promedio: 45.5 | mediana: 37
Ofertas con 0 tokens: 0


## 10 · Análisis de calidad

In [ ]:
# Top 50 skills más demandadas
all_tokens = [t for toks in df_total['habilidades'] for t in toks]
Counter(all_tokens).most_common(50)

[('datos', 1846),
 ('ingenieria', 1731),
 ('sistemas', 1662),
 ('desarrollo', 1351),
 ('avanzado', 981),
 ('intermedio', 941),
 ('sql', 797),
 ('gestion', 753),
 ('bases_datos', 744),
 ('procesos', 705),
 ('analisis', 702),
 ('software', 641),
 ('informatica', 635),
 ('power_bi', 597),
 ('minima', 594),
 ('python', 566),
 ('informacion', 514),
 ('trabajo', 506),
 ('industrial', 455),
 ('excel', 453),
 ('proyectos', 435),
 ('azure', 402),
 ('aplicaciones', 398),
 ('integracion', 368),
 ('diseno', 362),
 ('net', 348),
 ('programacion', 344),
 ('automatizacion', 338),
 ('apis', 331),
 ('tecnicos', 325),
 ('web', 320),
 ('comunicacion', 318),
 ('cloud', 318),
 ('estadistica', 310),
 ('administracion', 301),
 ('sql_server', 298),
 ('data', 290),
 ('java', 285),
 ('git', 274),
 ('ano', 273),
 ('universitario', 273),
 ('computacion', 269),
 ('servicios', 266),
 ('seguridad', 266),
 ('basico', 262),
 ('ia', 261),
 ('react', 256),
 ('calidad', 252),
 ('aws', 243),
 ('javascript', 241)]

In [ ]:
# Bigrams técnicos que se conservaron
bigrams_guardados = [(t,n) for t,n in freq_global.most_common(200) if '_' in t]
for t, n in bigrams_guardados[:10]:
    print(f"  {t:35s} {n}")

  bases_datos                         626
  power_bi                            543
  sql_server                          276
  api_rest                            201
  control_versiones                   126
  ci_cd                               110
  machine_learning                    84


## 11 · Guardar resultado

In [ ]:
output_cols = ['titulo', 'detalle', 'portal', 'ubicacion', 'Nivel', 'habilidades']
df_output = df_total[output_cols].copy()

# Guardar la lista como string (para compatibilidad con el pipeline)
df_output['habilidades'] = df_output['habilidades'].apply(str)

output_path = "df_ofertas_1.csv"
df_output.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")
print(f"Shape: {df_output.shape}")
df_output.head(3)

Guardado: df_ofertas_1.csv
Shape: (2087, 6)


,titulo,detalle,portal,ubicacion,Nivel,habilidades
0,Analista de Datos,¡Sé parte de Stefanini!​En Stefanini somos más...,bumeran,"San Borja, Lima",No especifica,"['control', 'automatizacion', 'sistemas']"
1,Analista de Datos (Educación),"Somos VISIVA (Certus,UCAL y Toulouse Lautrec) ...",bumeran,"San Borja, Lima",No especifica,"['datos', 'tls', 'funciones', 'arquitectura', ..."
2,Analista de Datos e Integración,¡Únete a Qorikallpa!Somos el corporativo de AP...,bumeran,"San Isidro, Lima",No especifica,"['datos', 'integracion', 'formar', 'excelencia..."
